<a href="https://colab.research.google.com/github/Srikara2005/Text-Classification/blob/main/TextClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Importing packages
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

In [ ]:
#Load IMDB Dataset
train_data, val_data, test_data = tfds.load(
    "imdb_reviews",
    split=("train[:60%]", "train[60%:]", "test"),
    as_supervised=True
)

In [ ]:
#Text Vectorization
max_tokens = 10000
sequence_length = 250

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_sequence_length=sequence_length
)

In [ ]:
# Fit vectorizer on training data
vectorizer.adapt(train_data.map(lambda x, y: x))

In [ ]:
#Build model
model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(input_dim=10000, output_dim=32),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
#Compile model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

In [ ]:
#Train model
history = model.fit(
    train_data.shuffle(10000).batch(512),
    epochs=15,
    validation_data=val_data.batch(512)
)

In [ ]:
#Evaluation on test data
results = model.evaluate(test_data.batch(512))
print(f"\nTest Loss: {results[0]:.4f}, Test Accuracy: {results[1]*100:.2f}%")

In [ ]:
#Prediction on new text
examples = [
    "The movie was amazing! I really loved it.",
    "Worst movie ever. Waste of time.",
    "It is a complete waste of time.",
    "It is a very good movie. It is an entertainer."
]
examples_dataset = tf.data.Dataset.from_tensor_slices(examples).batch(2)

predictions = model.predict(examples_dataset)

for review, score in zip(examples, predictions):
    print(f"Review: {review}")
    print(f"Sentiment score: {score[0]:.3f} ({'Positive' if score[0] > 0.1 else 'Negative'})\n")


In [ ]:
#Visualization of the model
plt.figure(figsize=(12,5))
# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy')

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss')

plt.show()